<a href="https://colab.research.google.com/github/sjsu-cs131-spring26/trendtrackers-engagement-socialmedia/blob/main/finaldemo_trendtrackers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1: Setup Project Variables**

In [ ]:
PROJECT_ID = "trendtrackers-final-cs131"
REGION = "us-west1"
BUCKET = "YOUR_BUCKET_NAME"

DATA_PATH = f"gs://{BUCKET}/data/exorde_dataset/"
OUTPUT_PATH = f"gs://{BUCKET}/output/final_sprint/"
CODE_PATH = f"gs://{BUCKET}/code/final_pipeline.py"

print("Project:", PROJECT_ID)
print("Bucket:", BUCKET)
print("Data path:", DATA_PATH)

# **2: Setup GCP**

In [ ]:
from google.colab import auth
auth.authenticate_user()

!gcloud config set project {PROJECT_ID}

# **3: PySpark Pipeline**

In [ ]:
pipeline_code = """
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, regexp_extract, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("CS131_Final_Project").getOrCreate()

input_path = "DATA_PATH_PLACEHOLDER"
output_path = "OUTPUT_PATH_PLACEHOLDER"

df = spark.read.parquet(input_path)

print("Schema:")
df.printSchema()

print("Row count:", df.count())

# basic cleaning
clean_df = df.filter(col("text").isNotNull())

# posts by language
language_counts = (
    clean_df.groupBy("lang")
    .agg(count("*").alias("post_count"))
    .orderBy(col("post_count").desc())
)

language_counts.write.mode("overwrite").parquet(output_path + "language_counts")

# posts by theme
theme_counts = (
    clean_df.groupBy("primary_theme")
    .agg(count("*").alias("post_count"))
    .orderBy(col("post_count").desc())
)

theme_counts.write.mode("overwrite").parquet(output_path + "theme_counts")

# sentiment by theme
sentiment_by_theme = (
    clean_df.groupBy("primary_theme")
    .agg(
        count("*").alias("post_count"),
        avg("sentiment_score").alias("avg_sentiment")
    )
)

sentiment_by_theme.write.mode("overwrite").parquet(output_path + "sentiment_by_theme")

# extract domain from url
domain_df = clean_df.withColumn(
    "domain",
    regexp_extract(col("url"), "https?://([^/]+)", 1)
)

top_domains = (
    domain_df.groupBy("domain")
    .agg(count("*").alias("post_count"))
    .orderBy(col("post_count").desc())
)

top_domains.write.mode("overwrite").parquet(output_path + "top_domains")

# window function example (top themes per language)
lang_theme = clean_df.groupBy("lang","primary_theme").count()

window = Window.partitionBy("lang").orderBy(col("count").desc())

top_themes = (
    lang_theme
    .withColumn("rank", row_number().over(window))
    .filter(col("rank") <= 3)
)

top_themes.write.mode("overwrite").parquet(output_path + "top_themes_per_language")

spark.stop()
"""

pipeline_code = pipeline_code.replace("DATA_PATH_PLACEHOLDER", DATA_PATH)
pipeline_code = pipeline_code.replace("OUTPUT_PATH_PLACEHOLDER", OUTPUT_PATH)

with open("final_pipeline.py","w") as f:
    f.write(pipeline_code)

print("Pipeline created.")

# **4: Pipeline to GCP**

In [ ]:
!gsutil cp final_pipeline.py {CODE_PATH}
!gsutil ls gs://{BUCKET}/code/

!gcloud dataproc batches submit pyspark {CODE_PATH} \
  --region={REGION} \
  --deps-bucket=gs://{BUCKET}

## Checkout outputs
!gsutil ls -r {OUTPUT_PATH}

# **5: Visualizations**

In [1]:
!pip install gcsfs pyarrow

In [ ]:
import pandas as pd

language_counts = pd.read_parquet(f"{OUTPUT_PATH}language_counts")
theme_counts = pd.read_parquet(f"{OUTPUT_PATH}theme_counts")
sentiment = pd.read_parquet(f"{OUTPUT_PATH}sentiment_by_theme")
domains = pd.read_parquet(f"{OUTPUT_PATH}top_domains")

## **Top Languages**

In [ ]:
import matplotlib.pyplot as plt

top_lang = language_counts.head(10)

plt.figure(figsize=(10,6))
plt.bar(top_lang["lang"], top_lang["post_count"])
plt.title("Top Languages in Social Media Posts")
plt.xlabel("Language")
plt.ylabel("Post Count")
plt.xticks(rotation=45)
plt.show()

## **Common Themes**

In [ ]:
top_themes = theme_counts.head(10)

plt.figure(figsize=(10,6))
plt.bar(top_themes["primary_theme"], top_themes["post_count"])
plt.title("Top Discussion Themes")
plt.xlabel("Theme")
plt.ylabel("Post Count")
plt.xticks(rotation=45)
plt.show()

## **Sentiment by Theme**

In [ ]:
sentiment_plot = sentiment.head(10)

plt.figure(figsize=(10,6))
plt.bar(sentiment_plot["primary_theme"], sentiment_plot["avg_sentiment"])
plt.title("Average Sentiment by Theme")
plt.xlabel("Theme")
plt.ylabel("Average Sentiment Score")
plt.xticks(rotation=45)
plt.show()